# Scheduler Simulation Dashboard

Interactive exploration notebook for `trace.json` files produced by the scheduler simulator.

## Setup

- Ensure the simulator has produced an up-to-date `trace.json` (default: `build/bin/Debug/results/trace.json`).
- This notebook assumes it is executed from the `python/` directory inside the repository.

In [2]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(NOTEBOOK_DIR))

import metrics_loader as ml

DEFAULT_TRACE = PROJECT_ROOT / "build" / "bin" / "Debug" / "results" / "trace.json"
trace_directory = DEFAULT_TRACE.parent

trace_files = sorted([path.name for path in trace_directory.glob("trace*.json")]) or [DEFAULT_TRACE.name]
trace_selector = widgets.Dropdown(options=trace_files, value=DEFAULT_TRACE.name, description="Trace:")
display(trace_selector)

trace = None
config = {}
summary = {}
tasks = None
ticks = None


Dropdown(description='Trace:', options=('trace.json', 'trace_fcfs.json', 'trace_mlfq.json'), value='trace.json…

In [11]:
selected_path = trace_directory / trace_selector.value
print(f'Using trace: {selected_path}')

trace = ml.load_trace(selected_path)
config = ml.config_dict(trace)
summary = ml.summary_dict(trace)

summary


Using trace: c:\Users\Andrew\Documents\Projects\Embedded-Systems-CPU-Scheduler-Simulator\build\bin\Debug\results\trace_mlfq.json


{'average_response_time_ms': 1.779940119760479,
 'average_runtime_ms': 2.530642750373692,
 'average_turnaround_time_ms': 5.389221556886228,
 'average_wait_time_ms': 2.8512705530642752,
 'completed_tasks': 1336,
 'core_idle_time_ms': [866, 1239],
 'cpu_utilization': {'average': 0.6368333333333334, 'samples': 3000},
 'simulation_end_ms': 3001,
 'simulation_start_ms': 0,
 'task_count': 1340,
 'total_idle_time_ms': 2105,
 'total_simulation_time_ms': 3001}

In [12]:
config

{'base_power_watts': 6.5,
 'clock_speed_mhz': 1000.0,
 'context_switch_cost_us': 50,
 'idle_power_watts': 1.2,
 'io_completion_quantum_us': 300,
 'max_power_watts': 15.0,
 'num_cores': 2,
 'planned_run_duration_ms': 3000,
 'system_name': '2 Core IoT Compute Node',
 'tick_interval_us': 500,
 'verbose_logging': True}

In [13]:
import pandas as pd

tasks = ml.task_lifecycle_df(trace)
ticks = ml.ticks_df(trace)

tasks.head()

,arrival_ms,class,completion_ms,deadline_ms,dispatch_count,final_state,first_dispatch_ms,name,priority,requested_exec_max_ms,requested_exec_min_ms,runtime_ms,task_id,turnaround_time_ms,wait_time_ms
0,2400,rt,2402,16,1,completed,2400,display_vsync#151,3,2,2,2,1052,2.0,0
1,490,rt,491,5,1,completed,490,audio_processing#99,2,1,1,1,399,1.0,0
2,0,rt,2,10,1,completed,0,wake_word_detection#1,1,2,2,2,0,2.0,0
3,10,rt,31,10,1,completed,28,wake_word_detection#2,1,3,3,3,1,21.0,18
4,672,rt,674,16,1,completed,672,display_vsync#43,3,2,2,2,944,2.0,0


## Core schedule

In [14]:
core_fig = ml.make_core_timeline_figure(trace)
core_fig

## CPU utilisation over time

In [15]:
cpu_fig = ml.make_cpu_utilisation_figure(trace, rolling_window=200)
cpu_fig

## Core idle totals

In [16]:
idle_fig = ml.make_core_idle_bar_figure(trace)
idle_fig

## Task runtime vs wait

In [17]:
runtime_fig = ml.make_task_runtime_scatter(trace)
runtime_fig

## Additional exploration

In [18]:
tasks.groupby("class")["runtime_ms"].sum().sort_values(ascending=False)

class
rt             2770
interactive     529
background       87
Name: runtime_ms, dtype: int64